In [89]:
import numpy as np
import pandas as pd

import scanpy as sc
from anndata import AnnData

In [90]:
base = "/data/a330d/projects/cellina-reproducibility/notebooks/in_vivo/data/"

df = pd.read_csv(f"{base}/merfishcounttable.csv", header=None)
cell_idx = df.iloc[:, 0].values
area     = df.iloc[:, 1].values
counts   = df.iloc[:, 3:3+500].values          # first 500 count cols = real genes; drop ~50 blanks

# gene names: prefer the codebook; fallback = tumorMerfish.csv column order
try:
    cb = pd.read_csv(f"{base}/codebook_0_ImmunOncology_0.csv")
    gene_names = cb.iloc[:, 0].astype(str).values[:500]   # adjust col if needed
except FileNotFoundError:
    gene_names = pd.read_csv(f"{base}/tumorMerfish.csv", nrows=0).columns[:500].values

In [91]:
total = counts.sum(1)
keep  = (total > 40) & (total < 800) & (area > 2000) & (area < 30000)   # strict, matches authors

coords = pd.read_csv(f"{base}/coordinates.csv", header=None).values      # check header=None vs 0

adata = AnnData(counts.astype(np.int32))
adata.var_names = gene_names

adata

AnnData object with n_obs × n_vars = 187215 × 500

In [92]:
mean_per_cell = np.asarray(adata.X.mean(axis=0)).ravel()
keep_genes = adata.var_names[mean_per_cell >= 0.2]
adata = adata[:, keep_genes].copy()
print(len(keep_genes))

154


In [93]:
# Filter adata to keep only cells that pass QC
adata = adata[keep]
adata.obs["cell_index"]  = cell_idx[keep]
adata.obs["area"]        = area[keep]
adata.obs["total_counts"] = total[keep]
adata.obsm["spatial"]    = coords[keep, -2:] 

adata

/tmp/ipykernel_3889525/441295888.py:3: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["cell_index"]  = cell_idx[keep]


AnnData object with n_obs × n_vars = 153705 × 154
    obs: 'cell_index', 'area', 'total_counts'
    obsm: 'spatial'

In [94]:
ref = sc.read_h5ad(f"{base}/tumors_qc_test.h5ad")   # 154418 × 154, already annotated

# coordinate lookup from the reference (round to kill float noise; drop rare dup coords)
ref_lookup = pd.DataFrame(np.asarray(ref.obsm["spatial"])[:, :2], columns=["x", "y"])
for c in ["celltype", "celltype2", "perturbation", "n_perturb"]:
    if c in ref.obs:
        ref_lookup[c] = ref.obs[c].values
ref_lookup = ref_lookup.round({"x": 3, "y": 3}).drop_duplicates(["x", "y"])

mine = pd.DataFrame(np.asarray(adata.obsm["spatial"])[:, :2], columns=["x", "y"]).round({"x": 3, "y": 3})
merged = mine.merge(ref_lookup, on=["x", "y"], how="left")

for c in ["celltype", "celltype2", "perturbation", "n_perturb"]:
    if c in merged:
        adata.obs[c] = merged[c].values

print("matched:", adata.obs["celltype"].notna().sum(), "/", adata.n_obs)

matched: 9371 / 153705


In [95]:
# What does the reference spatial actually look like?
rs = np.asarray(ref.obsm["spatial"])
print("ref spatial:", rs.shape, rs.dtype)
print("ref x range:", rs[:,0].min(), rs[:,0].max())
print("ref y range:", rs[:,1].min(), rs[:,1].max())
print(rs[:5])

# What does your coords file look like?
ms = np.asarray(adata.obsm["spatial"])
print("\nmine spatial:", ms.shape)
print("mine x range:", ms[:,0].min(), ms[:,0].max())
print("mine y range:", ms[:,1].min(), ms[:,1].max())
print(ms[:5])

# Raw coordinates.csv — how many columns, any header?
craw = pd.read_csv(f"{base}/coordinates.csv", header=None, nrows=5)
print("\ncoordinates.csv head:", craw.shape)
print(craw)

# Do the ROW COUNTS line up with anything?
print("\ncoordinates.csv full length:")
print(len(pd.read_csv(f"{base}/coordinates.csv", header=None)))

ref spatial: (154418, 2) float64
ref x range: 63.9312207555065 89393.1968057344
ref y range: 92974.7980258842 153159.954802474
[[7.03205403e+01 1.26665503e+05]
 [7.19550267e+01 1.28280027e+05]
 [1.26485493e+02 1.28631438e+05]
 [9.97899967e+01 1.29238802e+05]
 [1.28678001e+02 1.30309648e+05]]

mine spatial: (153705, 2)
mine x range: 63.9312207555065 89403.7582707895
mine y range: 93086.0898650348 153159.954802474
[[7.19550267e+01 1.28280027e+05]
 [6.39312208e+01 1.30905585e+05]
 [8.56270636e+01 1.32744086e+05]
 [8.01415702e+01 1.34249789e+05]
 [7.68264591e+01 1.34434331e+05]]

coordinates.csv head: (5, 2)
            0              1
0   75.164793  124907.488559
1   52.238455  125139.546426
2   89.813397  125482.203594
3   55.184152  126332.680584
4  112.873177  126514.208587

coordinates.csv full length:
187215


In [96]:
df = pd.read_csv(f"{base}/merfishcounttable.csv", header=None)
coords_all = pd.read_csv(f"{base}/coordinates.csv", header=None).values

print("counttable rows:", len(df), " coords rows:", len(coords_all))   # both 187215?

# The counttable col 3 is a linearized centroid (sub2ind over mask [102389,169973]).
# Decode it and compare to coordinates.csv to see if the two files share row order.
lin = df.iloc[:, 2].values.astype(np.int64)
# MATLAB sub2ind([nrows, ncols], r, c) = r + (c-1)*nrows, with nrows=102389
nrows = 102389
r = ((lin - 1) % nrows) + 1
c = ((lin - 1) // nrows) + 1
print("\ndecoded centroid (r,c) first 5:")
print(np.column_stack([r, c])[:5])
print("coordinates.csv first 5:")
print(coords_all[:5])

counttable rows: 187215  coords rows: 187215

decoded centroid (r,c) first 5:
[[34683   125]
 [28839    87]
 [67789   149]
 [18776    92]
 [97579   187]]
coordinates.csv first 5:
[[7.51647932e+01 1.24907489e+05]
 [5.22384549e+01 1.25139546e+05]
 [8.98133966e+01 1.25482204e+05]
 [5.51841517e+01 1.26332681e+05]
 [1.12873177e+02 1.26514209e+05]]


In [97]:
# Reference annotations keyed by exact coordinate
ref_df = pd.DataFrame(np.asarray(ref.obsm["spatial"])[:, :2], columns=["x", "y"])
for col in ["celltype", "celltype2", "perturbation", "n_perturb"]:
    if col in ref.obs:
        ref_df[col] = ref.obs[col].values

# All-cells coordinates tagged with original row index (== counttable row index)
coords_all = pd.read_csv(f"{base}/coordinates.csv", header=None).values
all_df = pd.DataFrame(coords_all[:, :2], columns=["x", "y"])
all_df["orig_row"] = np.arange(len(all_df))

hit = all_df.merge(ref_df, on=["x", "y"], how="inner").drop_duplicates("orig_row")
print("exact matches:", len(hit), "/", len(ref_df))

exact matches: 154418 / 154418


In [98]:
rows = hit["orig_row"].values                      # original counttable/coords rows
counts = df.iloc[rows, 3:3+500].values

adata = AnnData(counts.astype(np.int32))
adata.var_names = gene_names

In [99]:
adata.obs_names = [str(i) for i in rows]
adata.obs["cell_index"]   = df.iloc[rows, 0].values
adata.obs["area"]         = df.iloc[rows, 1].values
adata.obs["total_counts"] = counts.sum(1)
adata.obsm["spatial"]     = coords_all[rows, :2]
for col in ["celltype", "celltype2", "perturbation", "n_perturb"]:
    if col in hit:
        adata.obs[col] = hit[col].values

assert np.allclose(adata.X, np.round(adata.X))
print(adata)
print(adata.obs["celltype"].value_counts(dropna=False))
print(adata.obs["perturbation"].value_counts(dropna=False).head())

AnnData object with n_obs × n_vars = 154418 × 500
    obs: 'cell_index', 'area', 'total_counts', 'celltype', 'celltype2', 'perturbation', 'n_perturb'
    obsm: 'spatial'
celltype
NaN      145047
tumor      9371
Name: count, dtype: int64
perturbation
NaN        145047
Control      4250
IRF3          427
IRAK4         383
IRF5          362
Name: count, dtype: int64


In [100]:
adata = adata[:, keep_genes].copy()

In [101]:
adata.layers["counts"] = adata.X.copy()
sc.pp.filter_cells(adata, min_counts=3)

In [44]:
# Option 1: Subset to top-n genes by total counts
TOP_N = 150
gene_totals = np.asarray(adata.X.sum(axis=0)).ravel() 
top_genes = adata.var_names[np.argsort(-gene_totals)[:TOP_N]]

adata = adata[:, top_genes].copy()

In [ ]:
# Option 2: Subset to genes detected in x% of cells
pct_detected = np.asarray((adata.X > 0).mean(axis=0)).ravel()
keep_genes = adata.var_names[pct_detected >= 0.15]
adata = adata[:, keep_genes].copy()

In [ ]:
sc.pp.normalize_total(adata, target_sum=None) # Defaults to median
sc.pp.log1p(adata)
adata.layers["lognorm"] = adata.X
adata.X = adata.layers["counts"].copy()  # restore raw counts for top-n selection

In [103]:
adata

AnnData object with n_obs × n_vars = 154418 × 154
    obs: 'cell_index', 'area', 'total_counts', 'celltype', 'celltype2', 'perturbation', 'n_perturb', 'n_counts'
    uns: 'log1p'
    obsm: 'spatial'
    layers: 'counts', 'log1p'

In [104]:
# Check how many ref.var genes are in adata.var
print("ref.var genes:", len(ref.var_names))
print("adata.var genes:", len(adata.var_names))
print("common genes:", len(set(ref.var_names) & set(adata.var_names)))

ref.var genes: 154
adata.var genes: 154
common genes: 154


In [105]:
adata.write_h5ad(f"pfish_full.h5ad")